#  Customer Churn — Exploratory Data Analysis
**Dataset:** Telco Customer Churn (7043 rows × 21 columns)  
**Target:** `Churn` (binary — Yes/No → 1/0)  
**Churn rate:** ~26.5%

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml

# Project imports
from src.data.ingest import DataIngester

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
print('✅ Imports complete')

## 1. Load & Inspect

In [ ]:
ingester = DataIngester(config)
df = ingester.ingest_csv('../data/raw/churn_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic stats
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== NULL COUNTS ===')
print(df.isnull().sum())
print('\n=== CHURN DISTRIBUTION ===')
print(df['Churn'].value_counts())
print(f'Churn Rate: {df["Churn"].mean():.2%}')

## 2. Class Imbalance

In [ ]:
churn_counts = df['Churn'].value_counts().reset_index()
churn_counts.columns = ['Churn', 'Count']
churn_counts['Label'] = churn_counts['Churn'].map({1: 'Churned', 0: 'Stayed'})

fig = px.pie(
    churn_counts, values='Count', names='Label',
    title='Target Class Distribution',
    color_discrete_map={'Churned': '#ef4444', 'Stayed': '#22c55e'},
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()
print('⚠️  Imbalanced dataset — will use SMOTE + class weighting in training')

## 3. Numerical Feature Analysis

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig = make_subplots(rows=1, cols=3, subplot_titles=num_cols)
colors = {'0': '#22c55e', '1': '#ef4444'}

for i, col in enumerate(num_cols, 1):
    for churn_val in [0, 1]:
        subset = df[df['Churn'] == churn_val][col]
        fig.add_trace(
            go.Histogram(
                x=subset, name=f'{"Churned" if churn_val else "Stayed"}',
                opacity=0.65,
                marker_color='#ef4444' if churn_val else '#22c55e',
                showlegend=(i == 1)
            ),
            row=1, col=i
        )

fig.update_layout(
    barmode='overlay', title='Numerical Features by Churn Status',
    height=350, plot_bgcolor='rgba(0,0,0,0)'
)
fig.show()

print('\n📌 Key observations:')
print('  • Customers with lower tenure churn significantly more')
print('  • Higher monthly charges → higher churn risk')
print('  • TotalCharges correlated with tenure (longer = more total)')

## 4. Categorical Feature Analysis

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Churn Rate by Categorical Features', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), cat_cols):
    rates = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    bars = ax.bar(rates.index, rates.values,
                  color=['#ef4444' if v > 0.3 else '#6366f1' for v in rates.values])
    ax.set_title(col)
    ax.set_ylabel('Churn Rate')
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('\n📌 Key observations:')
print('  • Month-to-month contracts churn at ~43% vs 3% for 2-year')
print('  • Fiber optic users churn at ~41% — despite (or because of) premium cost')
print('  • Electronic check users churn most among payment methods')

## 5. Correlation Heatmap

In [ ]:
# Encode categoricals for correlation
df_enc = df.copy()
binary_map = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}
for col in df_enc.select_dtypes('object').columns:
    if col not in ['customerID', 'Contract', 'InternetService', 'PaymentMethod', 'gender']:
        df_enc[col] = df_enc[col].map(binary_map).fillna(0)

df_enc['gender']   = (df_enc['gender']   == 'Female').astype(int)
df_enc['Contract'] = df_enc['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df_enc = df_enc.drop(columns=['customerID', 'InternetService', 'PaymentMethod'])

corr = df_enc.select_dtypes('number').corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Top correlations with Churn
churn_corr = corr['Churn'].abs().sort_values(ascending=False).drop('Churn')
print('\nTop 10 features correlated with Churn:')
for feat, val in churn_corr.head(10).items():
    print(f'  {feat:25s}: {val:.3f}')

## 6. Tenure Analysis — The Most Important Feature

In [ ]:
# Churn rate by tenure bucket
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12m (New)', '13-24m', '25-48m', '49-72m (Loyal)']
)

tenure_churn = df.groupby('tenure_group')['Churn'].agg(['mean', 'count']).reset_index()
tenure_churn.columns = ['Tenure Group', 'Churn Rate', 'Count']

fig = px.bar(
    tenure_churn, x='Tenure Group', y='Churn Rate',
    text='Count', color='Churn Rate',
    color_continuous_scale='RdYlGn_r',
    title='Churn Rate by Tenure Group (count = # customers in group)'
)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.update_layout(yaxis_tickformat='.0%', height=350)
fig.show()

print('\n📌 Insight: New customers (0-12m) churn 3.5x more than loyal ones (49-72m)')
print('   → tenure_group is a critical engineered feature')